In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_debug_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe086.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_debug_export.log


In [9]:
input_phased_path = "data/mt/ukb_wes_200k_annotated_chr21.mt"
input_unphased_path = "data/mt/ukb_wes_200k_annotated_chr21_singletons.mt"

In [36]:
mt = qc.get_table(input_path=input_phased_path, input_type='mt') 
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt')

In [11]:
consequence_annotations = hl.read_table('data/vep/hail/ukb_wes_200k_chr21_vep.ht')
mt1 = mt1.annotate_rows(consequence = consequence_annotations[mt1.row_key]) 
mt2 = mt2.annotate_rows(consequence = consequence_annotations[mt2.row_key])

In [12]:
mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
mt2 = mt2.explode_rows(mt2.consequence.vep.worst_csq_by_gene_canonical)    

In [14]:
mt1.consequence.vep.worst_csq_for_variant_canonical.gene_id

<StringExpression of type str>

In [17]:
category_annotation_mt1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_for_variant_canonical, mt1.consequence.dbnsfp, use_loftee = False)
mt1 = mt1.annotate_rows(consequence_category = category_annotation_mt1)
category_annotation_mt2 = analysis.annotation_case_builder(mt2.consequence.vep.worst_csq_for_variant_canonical, mt2.consequence.dbnsfp, use_loftee = False)
mt2 = mt1.annotate_rows(consequence_category = category_annotation_mt2)




In [19]:
expr_gene_id = mt1.consequence.vep.worst_csq_for_variant_canonical.gene_id
analysis.count_urv_by_genes(mt1, expr_gene_id)


In [21]:
ht = hl.read_table('/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/200k/phenotypes.ht')

In [28]:
TRANCHE = '200k'
SEXCHECK_LIST = '/well/lindgren/UKBIOBANK/dpalmer/wes_' + TRANCHE + '/ukb_wes_qc/data/samples/04_sexcheck.remove.sample_list'
PHENOTYPES_TABLE = '/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/' + TRANCHE + '/phenotypes.ht'
IMPUTESEX_TABLE = '/well/lindgren/UKBIOBANK/dpalmer/wes_' + TRANCHE + '/ukb_wes_qc/data/samples/04_imputesex.ht'

sample_annotations = hl.read_table(PHENOTYPES_TABLE)
impute_sex_annotations = hl.read_table(IMPUTESEX_TABLE)
ht_sexcheck_samples = hl.import_table(SEXCHECK_LIST, no_header=True, key='f0')



2021-11-04 10:14:26 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)


In [53]:

#impute_sex_annotations.count()

mt.count()

#impute_sex_annotations.aggregate(hl.agg.sum(~hl.is_defined(impute_sex_annotations.is_female)))
#impute_sex_annotations.count()
#mt = mt.annotate_cols(is_female = impute_sex_annotations[mt.col_key].is_female)
#mt = mt.filter_cols(~hl.is_defined(ht_sexcheck_samples[mt.col_key]))
#x = mt.aggregate_cols(hl.agg.sum(~hl.is_defined(mt.is_female)))
#x

(60590, 185427)

In [80]:
# load imputed sex table
IMPUTESEX_TABLE = '/well/lindgren/UKBIOBANK/dpalmer/wes_' + TRANCHE + '/ukb_wes_qc/data/samples/04_imputesex.ht'
print(impute_sex_annotations.count())
print(impute_sex_annotations.aggregate(hl.agg.sum(~hl.is_defined(impute_sex_annotations.is_female))))


190792
0


In [81]:
# load phased data
mt = qc.get_table(input_path=input_phased_path, input_type='mt') 
print(mt.count())

(60590, 185506)


In [82]:
# subset to final samples
final_sample_list='/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/samples/09_final_qc.keep.sample_list'
final_samples =  ht_final_samples = hl.import_table(final_sample_list, no_header=True, key='f0', delimiter=',')
mt = mt.filter_cols(hl.is_defined(ht_final_samples[mt.col_key]))
mt.count()

2021-11-04 10:31:29 Hail: INFO: Loading 127 fields. Counts by type:
  str: 127


(60590, 176929)

In [85]:
mt = mt.annotate_cols(is_female = impute_sex_annotations[mt.col_key].is_female)
print(mt.aggregate_cols(hl.agg.sum(~hl.is_defined(mt.is_female))))

0


In [90]:
sex = '/well/lindgren/UKBIOBANK/dpalmer/wes_' + TRANCHE + '/ukb_wes_qc/data/samples/04_imputesex.tsv.bgz'
hl.import_table(sex).count()




2021-11-04 10:34:50 Hail: INFO: Loading 78 fields. Counts by type:
  str: 78


190792

In [95]:
ht1 = hl.read_table('/well/lindgren/UKBIOBANK/flassen/projects/KO/imp_ko_ukbb/data/samples/04_ukb_imp_500k_imputesex.ht')
ht2 = hl.import_table('/well/lindgren/UKBIOBANK/flassen/projects/KO/imp_ko_ukbb/data/samples/04_ukb_imp_500k_ycalled.tsv.bgz')


2021-11-04 11:25:18 Hail: INFO: Reading table without type imputation
  Loading field 's' as type str (not specified)
  Loading field 'n_called' as type str (not specified)


In [97]:
ht2.show()

,
s,n_called
str,str
"""-1""","""0"""
"""-10""","""0"""
"""-11""","""0"""
"""-12""","""579"""
"""-13""","""579"""
"""-14""","""0"""
"""-2""","""579"""
"""-3""","""577"""
